In [ ]:
import os, sys
_r = os.path.abspath(os.getcwd())
while _r != "/" and not os.path.exists(os.path.join(_r, "perm_paths.py")):
    _r = os.path.dirname(_r)
sys.path.insert(0, _r)
import perm_paths as P

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, P.FIGURES)
import figures as F

# ── THE cutoff for this notebook. Set it HERE, once. Every panel below reads
# F.TOP_N, and every file lands in panels/top<N>/ with its svg alongside.
# It is stored in the environment, so %autoreload cannot reset it mid-session.
F.use_top_n(300)              # the cutoff
F.use_label_mode("celltype")  # <- use this, NOT F.LABEL_MODE = "celltype"

# ── Mold brain: in or out of the SUPPLEMENT? ───────────────────────────────
# Figures 1 and 2 never show it. This switch only affects the panels below, and
# only the cells run AFTER it. Every panel resolves its datasets at call time.
SHOW_MOLD = False                  # <- True to put Mold brain back in the supplement
F.DATASET_EXCLUDE = frozenset() if SHOW_MOLD else frozenset({"Mold"})
F.clear_cache()                    # dataset-keyed caches, so drop them when this flips
print("supplement datasets:", F.active_datasets())


In [ ]:
# ===  page grid: one font size, exact-size export so the panels tile ===
F.SAVE_BBOX = None       # save at the exact figsize; set back to "tight" after Suppl. 1
FS, COL_W = 14, 4.6      # one grid column = 4.6 in
PAGE_W = 4 * COL_W       # 18.4 in
M_ROW = dict(left=.08, right=.965, wspace=.12)   # shared, so rows 2 and 3 get identical columns


In [ ]:
#---------- Suppl. fig 1 ----------

In [ ]:
# === Suppl. Fig 1a+b — SCG threshold (2 columns) ===

sw = F.load_scg_sweep("T-lineage_clones2-8")
fig_s1a, axes_s1a, thr_tab = F.fig_scg_threshold(
    sw=sw, canon=0.80,
    titles=("ClonePair count vs threshold", "Distribution of the continuous score"),
    legend=False,     panel_letters="alpha",        # a, b instead of i, ii
    colors={tp: F.C_OBS for tp in sw["timepoints"]},   # one red, as in Fig 1c
    linestyles=("-", (0, (2, 1))),                     # timepoint 1 solid, 2 dashed
    fs=FS, figsize=(2 * COL_W, 4.4),
    margins=dict(left=.10, right=.98, bottom=.17, top=.76, wspace=.32),
    save=f"{F.OUTDIR}/suppl1a_scg_threshold.png", dpi=400,
)


In [ ]:
# === Suppl. Fig 1c — what each method can supply (4 columns) ===
depth = F.ranking_depth_table(scg_variant="clones2-8_")
fig_s1b, axes_s1b, depth = F.fig_ranking_depth(
    table=depth, ncol=4, celltype=True,
    colors=[F.C_OBS, "white", "white"],        # one red, as in Fig 1c
    hatches=("", "xxx", "\\" * 3),             # solid | cross | backslash, per group
    hatch_color=F.C_OBS, hatch_lw=1.6, hatch_edge_lw=.8,
    annot_color=F.C_OBS,                       # counts in red, not grey
    panel_letters=None,                        # drops i, ii, iii
    pooled_legend=None,                        # the open diamond goes in the caption
    group_labels={"E13": "Day 13", "E14": "Day 14"},   # "embryonic" in the caption
    legend_ncol=2, headroom=400,               # two short rows instead of one wide one
    legend_fs=13, legend_handlelength=1.6, legend_handleheight=1.4,
    fs=FS, panel_w=COL_W, panel_h=4.0,
    margins={**M_ROW, "bottom": .30, "top": .80},
    save=f"{F.OUTDIR}/suppl1c_ranking_depth.png", dpi=400,
)


In [ ]:
# === Suppl. Fig 1d — cross-dataset sharing of memory genes across top-N cutoffs ===

# Fig 1d repeated at four cutoffs. Each panel is its own analysis: the consensus set,
# the shared-universe restriction and the expression-matched null are all rebuilt at
# that depth, so the panels are comparable in design but not nested in content.

fig_xc, axes_xc, stats_xc = F.fig_topn_sharing(
    cuts=tuple(c for c in (100, 300, 500, 1000) if c != F.TOP_N), fs=10, name="suppl1b_topn_sharing")
stats_xc.round(3)


In [ ]:
# === Suppl. Fig 1e+f — ortholog mapping | pairwise overlap, one figure ===
fig_s1d, axes_s1d, (ortho, res_s1e) = F.fig_ortholog_and_overlap(
    figsize=(PAGE_W, 4.2), fs=FS,
    margins=dict(left=.065, right=.915, bottom=.27, top=.86, wspace=.20),  # room for the longer colour-bar label
    cross_kw=dict(n_perm=1000, pair_universe=True,
                  matrix_metric="fold",          # colour = x more shared than chance
                  matrix_annot="n",              # squares keep the shared-gene count
                  cbar_ticks=(1, 5, 10)),        # 1 is marked "= chance" on the bar
    save=f"{F.OUTDIR}/suppl1e_ortholog_and_overlap.png", dpi=400,
)
F.SAVE_BBOX = "tight"    # back to normal for everything after Suppl. 1
res_s1e["pair_stats"].round({"expected": 1, "fold": 1})


In [ ]:
# === Suppl. Fig 1g — detection, protein, length: >=2 methods vs an EXPRESSION-MATCHED background ===

fig_s, axes_s, stats_s = F.fig_characterisation_matched(
    top_n=F.TOP_N, fs=10, name="suppl1g_characterisation")
stats_s.round(4)

In [ ]:
#---------- Suppl fig 2 ----------

In [ ]:
# === Suppl. Fig 2 panels rebuilt per group (each timepoint / arm as its own dataset) ===
# Per-group consensus sets, no merging across timepoints: one memory set per
# Michaels-T1/T2, Weinreb-2.0/4.0/6.0, Harmange-E/M, Bandler-E13/E14.
# sets_pg[unit] = (memory set at >= tier methods, every gene any method called)
sets_pg, summary_pg = F.pergroup_memory_sets()
summary_pg


In [ ]:
# === Suppl. Fig 2a — fig2b repeated, memory-gene vs non-memory knockouts, per group ===
# Built exactly like Fig 2b: every group is kept, the background is EVERY knockout no
# method called (one rule for the whole panel), and groups with fewer than MIN_KO
# memory-gene knockouts keep their grey half but show their knockouts as points and
# "too few (n = ...)" instead of a violin and statistics.
import matplotlib.pyplot as plt
import numpy as np

MIN_KO = 10                                   # same as Fig 2b
ORDER_3A = ("marson", "weinstock", "xu", "frangieh", "bowness", "mold")
K = F._register_pergroup_screens(F._ko_axis())
shorts = [K.SCREENS[s]["short"] for s in ORDER_3A]

res_3a, nmem_3a = {}, {}
for screen in ORDER_3A:
    folder = F.PERGROUP_SCREEN_FOLDER[screen]
    eff, M = K.load_screen(screen, metric="mean_abs_lfc")
    for g in F.PERGROUP_GROUPS[folder]:
        mem, called = sets_pg[f"{F.PERGROUP_LABEL[folder]}-{g}"]
        r = K.ko_axis_results(
            screen, top_n=F.TOP_N, eff=eff, M=M, metric="mean_abs_lfc",
            readout="all", control="random", n_iter=1,   # the draw is replaced below
            gene_sets={f">={F.PERGROUP_TIER}": mem}, exclude=called,
            min_memory=1, verbose=False)                  # keep every group
        if not r:
            continue
        short = K.SCREENS[screen]["short"]
        key = (f"{F.slabel(short, among=shorts)}\n{F.folder_glabel(folder, g)}"
               if F.LABEL_MODE == "celltype" else f"{short}\n{F.PERGROUP_LABEL[folder]}-{g}")
        v = list(r.values())[0]
        v["non_scg_values"] = F.ko_background_values(eff, M, set(mem) | set(called))
        res_3a[key] = v
        nmem_3a[key] = len(np.asarray(v["scg_values"]))
print("memory-gene knockouts per group:", nmem_3a)

import pandas as pd

# one row, full page width: 4.84 in (panel b) + 6.63 in (panels c,d) = 11.47 in
PAGE_W = 11.47
short_of = {K.SCREENS[s]["short"]: s for s in ORDER_3A}

def _screen_of(key):
    """Screen id of a panel label, in either label mode."""
    first = key.split("\n")[0]
    for short, s in short_of.items():
        if short in first or first in F.slabel(short, among=shorts):
            return s
    raise KeyError(key)

n = len(res_3a)
fig_b, ax = plt.subplots(1, 1, figsize=(PAGE_W, 3.6))
_, _, stats_b = F.fig_responsiveness_violin(
    res=res_3a, ax=ax,
    mem_label="Memory-gene knockouts", bg_label="Non-memory knockouts",
    annot="pct+stars", pct_where="left", annot_fs=8, pct_dx=4,
    show_medians=False, fill=False, median_inside=True, median_fmt="",
    xtick_rotation=45, xlabel="perturb-seq screen and group",
    ylabel="mean absolute log₂ fold change\nper knockout",
    legend=False, fs=7, show=False)
ax.set_xlim(-.6, n - .4)
F.mark_underpowered(ax, res_3a, nmem_3a, min_n=MIN_KO, label="too few\n(n = {n})")
# dividers AFTER mark_underpowered: it removes every line at the left edge of an
# underpowered column (x = i - 0.5), which is exactly where a divider sits
_scr = [_screen_of(k) for k in res_3a]
for i in range(len(_scr) - 1):
    if _scr[i] != _scr[i + 1]:
        ax.axvline(i + .5, color="0.8", ls=":", lw=.9, zorder=0)

fig_b.tight_layout()
F.save_fig(fig_b, f"{F.OUTDIR}/suppl2a_knockouts_per_group.png",
           "suppl2a_knockouts_per_group", dpi=400)
plt.show()
stats_b



In [ ]:
# === Suppl. Fig 2b — fig2d repeated within each experimental group of marson ===
import pandas as pd
fig_e, axes_e, stats_e = F.fig_marson_rungs_per_group(name="suppl2b_marson_rungs_per_group")
pd.concat(stats_e, names=["unit"]).round(3)


In [ ]:
# === Suppl. Fig 2c — Responsiveness survives every background-matching scheme, for every caller ===
# 183 x 132 mm: 3.6 in per column x 2 + 2.6 in per row x 2
# rung labels wrapped: at 91 mm per panel "state specificity" runs into its neighbours
PANELS_S5 = {
    "expression":         ["expr"],
    "detection":          ["detection_rate"],
    "state\nspecificity": ["state_restriction"],
    "gene\nlength":       ["log_len"],
    "all four":           ["expr", "detection_rate", "state_restriction", "log_len"],
}

fig_g, axes_g, res_g, stats_g = F.fig_matching_grid(
    panels=PANELS_S5,
    timepoint="merge", top_n=F.TOP_N,
    n_iter=2000, k=10, floor=F.EXPR_FLOOR,
    annot="pct+stars", ylim=(0, 1.0),
    fs=7, ncol=2, panel_w=3.6, panel_h=2.6,
    legend=False,
    title_fs=7, title_weight="normal", title_pad=14,
    xlabel_fs=6, ylabel_fs=6,
    save=f"{F.OUTDIR}/suppl2c_matching_grid.png", dpi=400,
)
stats_g.pivot(index="memory_set", columns="method", values="pct").round(0)


In [ ]:
#---------- Suppl fig 3 ----------

In [ ]:
# === Suppl. Fig 3a — memory genes have enough same expression-matched background genes ===

sup = F.matching_supply_table(top_ns=(100, 300, 500), min_methods=2)     # ~3 min
fig_sp, axes_sp, sup = F.fig_matching_supply(
    table=sup, kind="hist", ref_top_n=F.TOP_N, ncol=3, panel_w=2.0, panel_h=1.7,
    mem_colors=F.C_OBS,          # one red everywhere, not the per-dataset sunburst colours
    annot_smd=False,             # drop the |SMD| line from the corner annotation
    fs=7, save=f"{F.OUTDIR}/suppl3a_matching_supply.png", dpi=400,
)


In [ ]:
# === Suppl. Fig 3b - one forest plot for both sweeps, >=2-method consensus ===

T = F.robustness_tables(min_methods=2,          # the Fig 2e/2f gene set, not SCG alone
                        timepoint="merge", top_n=F.TOP_N,
                        floors=(None, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0),
                        n_bins=50, n_iter=100, n_boot=300, verbose=False)

fig_r, D = F.fig_robustness_forest(
    tables=T, min_methods=2,     # also used for the title
    ref_floor=F.EXPR_FLOOR,      # marked "←used" and drawn bold
    figsize=(4.0, 4.4), fs=7.5,
    group_labels=("Expression cutoff (UMI)", "Responsiveness metric"),
    annotate_n=True,             # ratio printed at the right edge of each row
    save=f"{F.OUTDIR}/suppl3b_robustness_forest.png", dpi=400,
)
D            # every row in the plot, as a tidy table for the supplement


In [ ]:
# === Suppl. Fig 3c — marson responsiveness per donor ===

donor_tab = F.donor_responsiveness(groups=F.MARSON_DONORS)

fig_d, ax_d, donor_tab, donor_sum = F.fig_donor_replication(
    table=donor_tab, value="pct", fs=15, orient="v",
    donor_palette="viridis", palette_range=(0., .80),
    dot_size=95, show_mean=False,
    legend_loc="upper left", legend_bbox=(.0, 1.0), legend_ncol=2,
    legend_fs=11,
    legend_frame=True, legend_edge="0.8", annot_weight="normal",
    ci_color="0.45", ref_label=None, note="",
    figsize=(6.6, 4.8), save=f"{F.OUTDIR}/suppl3c_donor_replication.png", dpi=400,
)


In [ ]:
#---------- Suppl fig 4 ----------

In [ ]:
# === Suppl. Fig 4a — responsiveness per screen x lineage group, heatmap ===
# same conventions as Fig 2g: stars over the cell values, no colour frames on the
# row names. The agreement lines are Suppl. Fig 4c (next cells).
fig_g, (ax_g, _), M_g = F.fig_crossdataset_per_group(
    name="suppl4a_crossdataset_per_group",
    lines=False,                            # heatmap only; lines are 4c
    label_box=False, label_chip=False, label_air_pt=2,
    annot_fmt="{:.0f}%", annot_stars=True,
    cbar_ticks=(),
    cbar_labels=("responsive", "less\nresponsive"),
    cbar_zero_label="0: no difference",
    cbar_below_label="negative values:\nless response",   # this panel HAS negatives
    cbar_zero_linespacing=1.15,
    cbar_text_rotation=90, cbar_rotate="all",
    heat_w_in=1.45, cbar_room_in=.62,
)
M_g.round(0)

In [ ]:
# === Suppl. Fig 4b — cross-dataset responsiveness sweep (full page width) ===
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

_g = plt.get_cmap("Greys")
BW = ListedColormap(np.vstack([_g(np.zeros(256)), _g(np.linspace(0, 1, 256))]))

DS = F.xd_datasets()
SHORTS_6 = [F._xd_pair_parts(d["name"])[1] for d in DS]

S_full = F.xd_sweep(DS, top_ns=[100, 250, 500, 1000, 2000], floor=F.EXPR_FLOOR)
S_full = S_full.replace({"method": {"POWERSEEK": "Power-Seek"}})   # house spelling

fig3, ax3, _ = F.fig_cross_dataset_sweep(
    kind="full", S=S_full, datasets=DS, top_ns=[100, 250, 500, 1000, 2000],
    methods=["CORAL", "GEMLI", "Power-Seek", "ClonePair"],
    metric="pct", group="method", show_n=False, cmap=BW,
    annot_fmt="{:.0f}%", annot_fs=7,                      # percentages 7 pt
    FS=10, fs_ticks=6, fs_rows=6, fs_header=6, fs_xlabel=6, cbar_fs=6,
    title_pad=2,                                          # kills the empty band on top
    cell=(0.30, 0.36), pad=(1.2, 0.55),                   # 183 x 69 mm
    cbar_label="", cbar_ticks=(),
    cbar_top_label="more\nresponsive",
    cbar_zero_label="0 = no\ndifference",
    cbar_below_label=None,                                # "less response" dropped
    cbar_text_rotation=90,
    save=None, show=False,
)
ax3.set_yticklabels(
    [F.slabel(F._xd_pair_parts(d["name"])[1], among=SHORTS_6, fmt="{ct} ({author})")
     for d in DS], fontsize=6)
F.save_fig(fig3, f"{F.OUTDIR}/suppl4b_cross_dataset_sweep.png",
           "suppl4b_cross_dataset_sweep", dpi=400)


In [ ]:
# === Suppl. Fig 4c — consensus tiers, as a normal line plot ===
import numpy as np, matplotlib.pyplot as plt
S_con, S_cut, row_labels = F.fig_crossdataset_per_group(return_tables=True)  # cached

fig_d, ax_d, _ = F.fig_cross_dataset_lines(
    S=S_con, tiers=(2, 3, 4), metric="pct",
    colors={r: F.DATASET_COLORS.get(F.dkey(F._xd_pair_parts(r)[0]), "0.5")
            for r in S_con.dataset.unique()},
    lw=.8, ms=3.0, line_alpha=.45, show_n=False,
    summary=True, summary_stat="mean", summary_lw=1.8, summary_ms=4.5,
    fs=7, figsize=(3.4, 2.3), legend=False,
    xlabel="Methods calling the gene",
    ylabel="% higher median response",
    save=None, show=False, verbose=False)      # saved below, after the edits

ax_d.set_xticklabels(["≥2", "≥3", "4"])        # at 4 every method agrees
_o = [">=2", ">=3", ">=4"]
_agg = np.nanmean(np.vstack(
    [S_con[S_con.dataset == r].set_index("method").reindex(_o)["pct"].to_numpy(float)
     for r in S_con.dataset.unique()]), axis=0)
for _xi, (_v, _p) in enumerate(zip(_agg, ["above", "below", "above"])):
    _dy, _va = (5, "bottom") if _p == "above" else (-5, "top")
    ax_d.annotate(f"{_v:.0f}%", (_xi, _v), xytext=(0, _dy),
                  textcoords="offset points", ha="center", va=_va,
                  fontsize=6, fontweight="bold", zorder=6)
F.save_fig(fig_d, f"{F.OUTDIR}/suppl4c_consensus_lines.png",
           "suppl4c_consensus_lines", dpi=400)
plt.show()


In [ ]:
# === Suppl. Fig 4d — leavout curve per method ===

lfc_df = F.load_marson_lfc()          # load once, reused across methods

fig_s, axes_s, leaveout_all = F.fig_leaveout_grid(
    lfc_df=lfc_df, top_n=F.TOP_N, step=1, x="n", ncol=2,   # CHANGED: every perturbation, as in Fig 2e
    panel_w=3.4, panel_h=2.7,
    xticks=(0, 3000, 6000), yticks=(.12, .20, .24), fs=10,
    annot_dy=(0, 12),
    fill_alpha=0,                                      # no shaded area
    legend_panel=None,                                 # no legend in any panel
    xlabel="number of perturbations removed",
    save=f"{F.OUTDIR}/suppl4d_leaveout_grid.png", dpi=400,
)


In [ ]:
#---------- Suppl fig 5 ----------

In [ ]:
# === Suppl. Fig 5a — overlap with bulk MemorySeq (left of row 2, 70 mm) ===
fig_ov, ax_ov, overlap_ms = F.fig_memoryseq_overlap(
    fs=7, figsize=(2.76, 2.75), xlabel="methods calling the gene",
    name="suppl5a_memoryseq_overlap")
overlap_ms.round(3)


In [ ]:
# === Suppl. Fig 5b — responsiveness of both sets (right of row 2, 110 mm) ===
fig_ms, axes_ms, stats_ms = F.fig_memoryseq_responsiveness(
    fs=7, figsize=(4.33, 2.75),
    labels={"Harmange": "Harmange\n(lineage)", "Shaffer": "Shaffer\n(bulk)"},
    title_fmt="{screen}", legend=False,
    name="suppl5b_memoryseq_responsiveness")
stats_ms[["screen", "method", "n_mem", "median_mem", "median_bg", "pct", "cliffs_d", "q", "smd_expr"]]


In [ ]:
#---------- Suppl fig 6 ----------

In [ ]:
# === Suppl. Fig 6a — responsiveness of memory genes found WITHOUT lineage barcodes ===
# Power-Seek, GEMLI and isCORAL, each on that screen's own control cells next to the same
# method on the cell-type-matched barcoded dataset (isCORAL's barcode partner = CORAL),
# plus the genes all three methods call on the control cells. All seven sets are scored
# on the SAME readout (that screen's knockout |LFC|), matched on the same control
# expression, floor 0.1, Figure-2 rank merge.
# === Suppl. Fig 6a — responsiveness of memory genes found WITHOUT lineage barcodes ===
fig6a, axes6a, stats6a, resp6a, panels6a = F.fig_lineage_free_responsiveness(
    panels=None, ylim=(0, 1.0), fs=7,
    median_weight="normal",      # medians inside the violins no longer bold
    ytick_fs=6,                  # axis numbers smaller than the rest
    pct_where="above",           # percentages above the violins, not inside
    col_title_pad=26,            # keeps the method names clear of them
    figsize=(7.2, 6.5),          # 183 x 165 mm
    name="suppl6a_control_vs_barcode_responsiveness", save=True, dpi=400,
)
stats6a[["screen", "column", "gene_set", "n_mem", "pct", "q"]].round({"pct": 1})


In [ ]:
# === Suppl. Fig 6b — summary of 6a: every estimate on one axis ===
fig6b, ax6b, summ6 = F.fig_lineage_free_summary(
    stats6a, row_gap=.30, fs=7,
    annot_fs=6, legend_fs=7, tick_fs=6,
    figsize=(7.2, 3.9),          # 183 x 99 mm
    name="suppl6b_lineage_free_summary", save=True, dpi=400,
)
summ6.round({"pct": 1})


In [ ]:
#---------- Suppl fig 7 ----------

In [ ]:
# === Suppl. Fig 7 — Power-Seek memory genes from control cells: responsiveness in every screen ===
res_ps, q_ps, table_ps = F.powerseek_responsiveness()      # Zhu is excluded: it is in Figure 2

fig_ps, (ax_ps, _), S_ps = F.fig_powerseek_summary(
    res=res_ps, q=q_ps, table=table_ps,
    panels="medians", orient="v",          # datasets on x, the metric on y
    sort_by="pct_higher",                  # strongest first, within each block
    annot_fmt="{pct:+.0f}%\n{stars}",      # percentage, stars underneath
    annot_rotation=0, annot_row=True,      # one row of labels, not chasing the markers
    ms=16, lw=.9,
    col_w=(7.2 - 1.6) / 18,                # 18 columns -> 7.2 in = 183 mm wide
    fig_h=3.6, fs=7, pad_right=.24,
    legend_loc="lower left",
    name="suppl7_powerseek_responsiveness",
    save=True, dpi=400,
)
table_ps.round(3)


In [ ]:
#---------- Suppl fig 8 ----------

In [ ]:
# === Suppl. Fig 8 — structure of perturbation responses: compute everything ===
box = F.box_compute()

In [ ]:
# === Suppl. Fig 8 — full figure, panels a–g ===
fig_box, _ = F.fig_box(data=box, name="suppl_box_response_structure", save=True, dpi=600)

In [ ]:
# === Suppl. Fig 8a — rank-one RMS factorisation (Bowness) ===
F.fig_box_panel("a", data=box, name="suppl_box_a", dpi=600)

In [ ]:
# === Suppl. Fig 8b — C, G_S/G_F, H_S/H_F across screens ===
F.fig_box_panel("b", data=box, name="suppl_box_b", dpi=600)
box["ghc"].round(3)

In [ ]:
# === Suppl. Fig 8c — first-mode fractions ===
F.fig_box_panel("c", data=box, name="suppl_box_c", dpi=600)
box["first_mode_tests"]

In [ ]:
# === Suppl. Fig 8d — held-out RMS prediction ===
F.fig_box_panel("d", data=box, name="suppl_box_d", dpi=600)
box["pred_by_screen"].round(3)

In [ ]:
# === Suppl. Fig 8e — clustered perturbation-response correlations (Bowness) ===
F.fig_box_panel("f", data=box, name="suppl_box_f", dpi=600)

In [ ]:
# === Suppl. Fig 8f — spectral scaling example ===
F.fig_box_panel("f", data=box, name="suppl_box_f", dpi=600)

In [ ]:
# === Suppl. Fig 8g — paired spectral slopes across screens ===
F.fig_box_panel("g", data=box, name="suppl_box_g", dpi=600)
box["paired_slopes"].round(3)

In [ ]:
# === Suppl. Fig 8 — manuscript numbers (what Ben's notebook printed) ===
bs = F.box_stats(box)
for k, v in bs.items():
    print(f"\n--- {k} ---")
    print(v.round(4).to_string(index=False) if hasattr(v, "to_string") else v)